In [ ]:
import pandas as pd
import requests
import time
import json
import os
# Add your team's loader here (adjust the import path if necessary)
from dataset import load_deepdta_dataset

# Ensure the splits folder exists in your data directory
os.makedirs('../data/splits', exist_ok=True)

def process_dataset(dataset_name):
    print(f"========================================")
    print(f"LOADING {dataset_name} DATASET")
    print(f"========================================")
    
    # Load Data
    data = DTI(name=dataset_name)
    df = data.get_data()
    
    # Deliverable 1: Basic Stats
    num_drugs = df['Drug'].nunique()
    num_targets = df['Target'].nunique()
    num_pairs = len(df)
    min_affinity = df['Y'].min()
    max_affinity = df['Y'].max()
    
    print(f"Total Pairs: {num_pairs}")
    print(f"Unique Drugs: {num_drugs}")
    print(f"Unique Proteins: {num_targets}")
    print(f"Affinity Range: {min_affinity:.4f} to {max_affinity:.4f}\n")
    
    # Deliverable 2: Build the four splits
    print(f"Generating splits for {dataset_name} (This may take a moment)...")
    splits = {
        'random': data.get_split(),
        'cold_drug': data.get_split(method='cold_split', column_name='Drug'),
        'cold_target': data.get_split(method='cold_split', column_name='Target'),
        'cold_pair': data.get_split(method='cold_split', column_name=['Drug', 'Target'])
    }
    
    # Save all splits to the data/splits folder
    for split_name, split_dict in splits.items():
        for fold in ['train', 'valid', 'test']:
            file_path = f"../data/splits/{dataset_name.lower()}_{split_name}_{fold}.csv"
            split_dict[fold].to_csv(file_path, index=False)
            
    print(f"All 12 CSV files saved for {dataset_name}.\n")
    
    # Deliverable 2: Sanity Checks (Crucial Step)
    print(f"Running Sanity Checks for {dataset_name}...")
    for split_name in ['cold_drug', 'cold_target', 'cold_pair']:
        train_df = splits[split_name]['train']
        test_df = splits[split_name]['test']
        
        if 'drug' in split_name or 'pair' in split_name:
            overlap = set(train_df['Drug']).intersection(set(test_df['Drug']))
            print(f"[{split_name}] Drug overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: DRUG LEAKAGE DETECTED <<<")
                
        if 'target' in split_name or 'pair' in split_name:
            overlap = set(train_df['Target']).intersection(set(test_df['Target']))
            print(f"[{split_name}] Target overlap between train/test: {len(overlap)}")
            if len(overlap) > 0: print(">>> WARNING: TARGET LEAKAGE DETECTED <<<")
    print("\n")

# Run the process for both datasets
process_dataset('DAVIS')
process_dataset('KIBA')

Downloading...


LOADING DAVIS DATASET


100%|██████████| 21.4M/21.4M [00:49<00:00, 429kiB/s] 
Loading...
Done!


Total Pairs: 25772
Unique Drugs: 68
Unique Proteins: 379
Affinity Range: 0.0160 to 10000.0000

Generating splits for DAVIS (This may take a moment)...


Downloading...


All 12 CSV files saved for DAVIS.

Running Sanity Checks for DAVIS...
[cold_drug] Drug overlap between train/test: 0
[cold_target] Target overlap between train/test: 0
[cold_pair] Drug overlap between train/test: 0
[cold_pair] Target overlap between train/test: 0


LOADING KIBA DATASET


100%|██████████| 96.6M/96.6M [03:20<00:00, 480kiB/s] 
Loading...
Done!


Total Pairs: 117657
Unique Drugs: 2068
Unique Proteins: 229
Affinity Range: 0.0000 to 17.2002

Generating splits for KIBA (This may take a moment)...
All 12 CSV files saved for KIBA.

Running Sanity Checks for KIBA...
[cold_drug] Drug overlap between train/test: 0
[cold_target] Target overlap between train/test: 0
[cold_pair] Drug overlap between train/test: 0
[cold_pair] Target overlap between train/test: 0




In [7]:
import pandas as pd
import re
import os

file_path = '../data/raw/BindingDB_All.tsv'
output_path = '../data/raw/bindingdb_antiviral.csv'

# The exact keywords requested by your guide
keywords = [
    r"sars-cov-2 main protease", r"mpro", r"3c-like proteinase", 
    r"sars-cov-2 rna-dependent rna polymerase", r"rdrp", 
    r"hiv-1 protease", r"hiv-1 reverse transcriptase", 
    r"influenza neuraminidase"
]
pattern = '|'.join(keywords)

print(f"Reading BindingDB in chunks to protect RAM. This will take a few minutes...")

# Read the massive file in chunks of 50,000 rows
chunk_iterator = pd.read_csv(file_path, sep='\t', chunksize=50000, on_bad_lines='skip', low_memory=False, dtype=str)

first_chunk = True
total_found = 0

for i, chunk in enumerate(chunk_iterator):
    # Find the target name columns dynamically
    target_cols = [c for c in chunk.columns if 'Target Name' in c]
    if not target_cols:
        target_cols = chunk.columns # Fallback to search all columns
        
    mask = pd.Series(False, index=chunk.index)
    for col in target_cols:
        mask = mask | chunk[col].fillna('').str.contains(pattern, flags=re.IGNORECASE, regex=True)
        
    filtered = chunk[mask]
    total_found += len(filtered)
    
    if len(filtered) > 0:
        filtered.to_csv(output_path, mode='a', index=False, header=first_chunk)
        first_chunk = False
        
    if i % 10 == 0 and i > 0:
        print(f"Processed {i * 50000} rows... found {total_found} matches so far.")

print(f"\nExtraction complete! Found a total of {total_found} antiviral pairs.")
print(f"Saved deliverable to: {output_path}")

Reading BindingDB in chunks to protect RAM. This will take a few minutes...
Processed 500000 rows... found 111 matches so far.
Processed 1000000 rows... found 139 matches so far.
Processed 1500000 rows... found 262 matches so far.
Processed 2000000 rows... found 262 matches so far.
Processed 2500000 rows... found 262 matches so far.
Processed 3000000 rows... found 613 matches so far.

Extraction complete! Found a total of 614 antiviral pairs.
Saved deliverable to: ../data/raw/bindingdb_antiviral.csv


In [8]:
import pandas as pd
from tdc.multi_pred import DTI

print("Loading datasets...")
davis = DTI(name='DAVIS').get_data()
kiba = DTI(name='KIBA').get_data()
bdb = pd.read_csv('../data/raw/bindingdb_antiviral.csv', low_memory=False)

# Extract unique targets
davis_targets = davis[['Target_ID', 'Target']].drop_duplicates()
kiba_targets = kiba[['Target_ID', 'Target']].drop_duplicates()

print(f"Unique DAVIS targets: {len(davis_targets)}")
print(f"DAVIS ID sample: {davis_targets['Target_ID'].head(3).tolist()}")

print(f"\nUnique KIBA targets: {len(kiba_targets)}")
print(f"KIBA ID sample: {kiba_targets['Target_ID'].head(3).tolist()}")

# BindingDB usually has an explicit UniProt column, let's find it
uniprot_cols = [c for c in bdb.columns if 'UniProt' in c]
if uniprot_cols:
    bdb_uniprot = uniprot_cols[0]
    print(f"\nBindingDB UniProt column found: '{bdb_uniprot}'")
    unique_bdb = bdb[bdb_uniprot].dropna().unique()
    print(f"Unique BindingDB targets: {len(unique_bdb)}")
    print(f"BindingDB ID sample: {unique_bdb[:3].tolist()}")
else:
    print("\nCould not find a clear UniProt column in BindingDB.")

Found local copy...
Loading...
Done!
Found local copy...
Loading...


Loading datasets...


Done!


Unique DAVIS targets: 379
DAVIS ID sample: ['AAK1', 'ABL1p', 'ABL2']

Unique KIBA targets: 229
KIBA ID sample: ['O00141', 'O14920', 'O15111']

BindingDB UniProt column found: 'UniProt (SwissProt) Recommended Name of Target Chain 1'
Unique BindingDB targets: 0
BindingDB ID sample: []


In [ ]:
import pandas as pd
import requests
import time
import json
import os
from tdc.multi_pred import DTI

print("--- PHASE 1: UNIFYING TARGET IDS ---")

# 1. KIBA Targets (Canonical DeepDTA Source)
kiba_data = load_deepdta_dataset('kiba')
kiba_ids = kiba_data['Target_ID'].dropna().unique().tolist()
print(f"KIBA: {len(kiba_ids)} UniProt IDs ready.")

# 2. BindingDB Targets 
bdb = pd.read_csv('../data/raw/bindingdb_antiviral.csv', low_memory=False)
bdb_uniprot_cols = [c for c in bdb.columns if 'uniprot' in c.lower()]
bdb_ids = set()

for col in bdb_uniprot_cols:
    for val in bdb[col].dropna().astype(str):
        tokens = [t.strip() for t in val.replace(';', ',').split(',')]
        for token in tokens:
            if len(token) >= 6 and token[0].isalpha():
                bdb_ids.add(token)

bdb_ids = list(bdb_ids)
print(f"BindingDB: {len(bdb_ids)} UniProt IDs extracted.")

# 3. DAVIS Targets (Canonical DeepDTA Source)
davis_data = load_deepdta_dataset('davis')
davis_genes = davis_data['Target_ID'].dropna().unique().tolist()
print(f"DAVIS: Mapping {len(davis_genes)} gene/protein names via UniProt API...")

davis_mapping_dict = {}
davis_mapped_ids = []
search_url = "https://rest.uniprot.org/uniprotkb/search"

for i, gene in enumerate(davis_genes):
    clean_gene = gene.rstrip('p').rstrip('m') 
    
    params = {
        "query": f"(gene:{clean_gene}) AND (taxonomy_id:9606)",
        "fields": "accession",
        "size": 1
    }
    
    try:
        r = requests.get(search_url, params=params, timeout=10)
        if r.status_code == 200:
            results = r.json().get('results', [])
            if results:
                # FIXED: The API returns 'primaryAccession', not 'accession'
                uid = results[0].get('primaryAccession')
                if uid:
                    davis_mapping_dict[gene] = uid
                    davis_mapped_ids.append(uid)
        else:
            if i == 0:
                print(f"  [Debug] API Error on '{gene}': {r.status_code} - {r.text}")
    except Exception as e:
        if i == 0:
            print(f"  [Debug] Request failed on '{gene}': {e}")
        pass
        
    time.sleep(0.1)

print(f"DAVIS: Mapped {len(davis_mapped_ids)} / {len(davis_genes)} gene targets.")

# Compile master set
all_uniprot_ids = list(set(kiba_ids + bdb_ids + davis_mapped_ids))
print(f"\nTotal Unified Unique UniProt IDs: {len(all_uniprot_ids)}")

print("\n--- PHASE 2: FETCHING GROUND TRUTH ---")

def get_binding_sites(uniprot_id):
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            sites = []
            for feature in data.get('features', []):
                if feature['type'] in ('Binding site', 'Active site'):
                    sites.append(feature['location'])
            return sites
    except Exception:
        pass
    return []

ground_truth = {}
print("Fetching site annotations from UniProt...")

for i, uid in enumerate(all_uniprot_ids):
    sites = get_binding_sites(uid)
    if sites:
        ground_truth[uid] = sites
        
    if (i + 1) % 50 == 0 or (i + 1) == len(all_uniprot_ids):
        print(f"  ...processed {i + 1} / {len(all_uniprot_ids)} targets")
    time.sleep(0.1)

print("\n--- PHASE 3: UPDATING DELIVERABLE ---")
out_path = '../data/processed/binding_sites_ground_truth.json'

final_payload = {
    "metadata": {
        "description": "Mapping of UniProt IDs to known Binding/Active site residue locations.",
        "source": "UniProt REST API",
        "davis_gene_to_uniprot_mapping": davis_mapping_dict,
        "total_targets_searched": len(all_uniprot_ids),
        "total_targets_with_sites": len(ground_truth)
    },
    "binding_sites": ground_truth
}

with open(out_path, 'w') as f:
    json.dump(final_payload, f, indent=2)

print(f"SUCCESS! Ground truth file updated at: {out_path}")
print(f"Total targets with binding site annotations: {len(ground_truth)}")

Found local copy...
Loading...


--- PHASE 1: UNIFYING TARGET IDS ---


Done!
Found local copy...
Loading...
Done!


KIBA: 229 UniProt IDs ready.
BindingDB: 10 UniProt IDs extracted.
DAVIS: Mapping 379 gene/protein names via UniProt API...
DAVIS: Mapped 342 / 379 gene targets.

Total Unified Unique UniProt IDs: 536

--- PHASE 2: FETCHING GROUND TRUTH ---
Fetching site annotations from UniProt...
  ...processed 50 / 536 targets
  ...processed 100 / 536 targets
  ...processed 150 / 536 targets
  ...processed 200 / 536 targets
  ...processed 250 / 536 targets
  ...processed 300 / 536 targets
  ...processed 350 / 536 targets
  ...processed 400 / 536 targets
  ...processed 450 / 536 targets
  ...processed 500 / 536 targets
  ...processed 536 / 536 targets

--- PHASE 3: UPDATING DELIVERABLE ---
SUCCESS! Ground truth file updated at: ../data/processed/binding_sites_ground_truth.json
Total targets with binding site annotations: 350
